---
## 1. Instalación y configuración del entorno

### ¿Qué hacemos aquí?
Instalamos y cargamos todas las librerías necesarias para esta entrega.
A diferencia de las entregas anteriores, aquí agregamos librerías
específicas para Machine Learning (scikit-learn) y conexión con
Google Cloud (google-cloud-bigquery).

### Librerías principales:
- **pandas / numpy** — manipulación y cálculo de datos
- **scikit-learn** — preprocesamiento y modelos de ML
- **matplotlib / seaborn** — visualización exploratoria
- **google-cloud-bigquery** — conexión con BigQuery para
  arquitectura Bronze/Silver/Gold

In [4]:
pip install pandas numpy matplotlib seaborn scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.0 MB 3.9 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.0 MB 3.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.0 MB 3.3 MB/s eta 0:00:02
   ------------- -------------------------- 2.6/8.0 MB 2.8 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.0 MB 3.1 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.0 MB 3.1 MB/s eta 0:00:02
   ----------------------- ---------------- 4.7/8.0 MB 3.1 MB/s eta 0:00:02
   ---------------------------- ----------- 5.8/8.0 MB 3.2 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.0 MB 3.3 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.0 MB 3.5 MB/s eta 0:00:01
   ------------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print(" Librerías cargadas correctamente")

 Librerías cargadas correctamente


---
Este notebook ejecuta el pipeline reproducible de semana 7 usando los modulos en `src/`. No crea formulas analiticas nuevas; compara estructuras de datos para Tableau con metricas de validacion estructural.

In [6]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT


WindowsPath('d:/Data_Visualization_TF/Data_Visualization_TF')

---
## 2. Carga del dataset

### ¿Qué hacemos aquí?
Cargamos el dataset Silver — es decir, el dataset ya limpio y
transformado que generamos en la Entrega 2. Este es el punto
de partida para el preprocesamiento específico del modelo.

### ¿Por qué Silver y no Bronze?
En la arquitectura Medallion:
- **Bronze** → dato crudo original (all_ai_models.csv)
- **Silver** → dato limpio (all_ai_models_clean.csv) ← estamos aquí
- **Gold** → dato final con clusters, listo para Tableau y BigQuery

No partimos del Bronze porque ya hicimos la limpieza en la
Entrega 2 y ese trabajo no se repite, se reutiliza.

In [7]:


from src.config import CLEAN_DATASET
from src.io_utils import load_csv

df = load_csv(CLEAN_DATASET)
df.shape

print(f"Dataset Silver cargado")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Periodo: {int(df['Year'].min())} – {int(df['Year'].max())}")
print(f"\nColumnas disponibles:")
for col in df.columns:
    print(f"  - {col}")

Dataset Silver cargado
Filas: 3405
Columnas: 22
Periodo: 1950 – 2026

Columnas disponibles:
  - Model
  - Organization
  - Country (of organization)
  - Publication date
  - Organization categorization
  - Domain
  - Task
  - Parameters
  - Training compute (FLOP)
  - Training dataset size (total)
  - Training time (hours)
  - Training compute cost (2023 USD)
  - Hardware quantity
  - Training hardware
  - Model accessibility
  - Training code accessibility
  - Open model weights?
  - Citations
  - Confidence
  - Notability criteria
  - Year
  - Month


---
## 3. Selección de variables para el modelo

### ¿Qué hacemos aquí?
No todas las variables del dataset Silver son útiles para PCA + K-Means.
Los modelos de ML no supervisado trabajan exclusivamente con variables
numéricas, las categóricas como Domain o Country no pueden entrar
directamente al modelo.

### ¿Qué variables necesita el modelo?
- Solo variables numéricas continuas
- Con cobertura suficiente, si una variable tiene 90% de nulos
  no aporta información útil al modelo
- Sin variables redundantes, si dos variables miden lo mismo
  una sobra

### Variables numéricas disponibles y su cobertura:
En la Entrega 2 identificamos que las variables numéricas tienen
coberturas muy distintas. Aquí evaluamos cuáles incluir.

In [8]:

numericas = ['Parameters', 'Training compute (FLOP)',
             'Training dataset size (total)',
             'Training time (hours)',
             'Training compute cost (2023 USD)',
             'Hardware quantity', 'Citations']

print("Cobertura de variables numéricas:")
print("="*50)
for col in numericas:
    total = len(df)
    con_dato = df[col].notna().sum()
    porcentaje = (con_dato / total * 100).round(1)
    print(f"{col}:")
    print(f"  Registros con dato: {con_dato} de {total} ({porcentaje}%)")
    print(f"  Registros sin dato: {total - con_dato} ({100-porcentaje}%)")
    print()

Cobertura de variables numéricas:
Parameters:
  Registros con dato: 2243 de 3405 (65.9%)
  Registros sin dato: 1162 (34.099999999999994%)

Training compute (FLOP):
  Registros con dato: 1366 de 3405 (40.1%)
  Registros sin dato: 2039 (59.9%)

Training dataset size (total):
  Registros con dato: 1379 de 3405 (40.5%)
  Registros sin dato: 2026 (59.5%)

Training time (hours):
  Registros con dato: 544 de 3405 (16.0%)
  Registros sin dato: 2861 (84.0%)

Training compute cost (2023 USD):
  Registros con dato: 224 de 3405 (6.6%)
  Registros sin dato: 3181 (93.4%)

Hardware quantity:
  Registros con dato: 838 de 3405 (24.6%)
  Registros sin dato: 2567 (75.4%)

Citations:
  Registros con dato: 1462 de 3405 (42.9%)
  Registros sin dato: 1943 (57.1%)



**Variables INCLUIDAS en el modelo:**
- Parameters (65.9%) → mejor cobertura, mide tamaño del modelo
- Training compute (FLOP) (40.1%) → mide potencia computacional
- Training dataset size (40.5%) → mide escala de entrenamiento
- Citations (42.9%) → mide impacto académico

**Variables EXCLUIDAS del modelo:**
- Training time (16.0%) → muy pocos registros, sesgaría el modelo
- Training compute cost (6.6%) → casi sin datos, no representativo
- Hardware quantity (24.6%) → baja cobertura y correlación alta
  con Training compute (0.79) — información redundante

**¿Por qué excluimos variables con baja cobertura?**
Si incluimos una variable con 6.6% de datos, el modelo solo podría
usar los 224 registros que tienen ese dato, perdemos el 93.4%
restante. Es mejor excluirla y trabajar con más registros usando
variables mejor documentadas.

**Variables finales para el modelo:** Parameters, Training compute
(FLOP), Training dataset size (total), Citations

---
## 3. Preprocesamiento para el modelo

### ¿Qué hacemos aquí?
Preparamos las variables que entrarán al modelo PCA + K-Means.
El modelo requiere exclusivamente variables numéricas. Las variables
categóricas se transformarán mediante One-Hot Encoding.

### Estrategia de variables:

**Numéricas reales (4):**
- Parameters → tamaño del modelo
- Training compute (FLOP) → potencia computacional  
- Training dataset size (total) → escala de entrenamiento
- Citations → impacto académico

**Categóricas con encoding (3):**
- Domain → tipo de IA (Language, Vision, Biology, etc.)
- Organization categorization → quién lo hizo (Industry, Academia)
- Model accessibility → qué tan abierto es el modelo

**¿Qué es One-Hot Encoding?**
Convierte cada categoría en una columna binaria (0 o 1).
Por ejemplo Domain = "Language" se convierte en:
- Domain_Language = 1
- Domain_Vision = 0
- Domain_Biology = 0

Se usa porque nuestras

In [10]:
# ============================================================
# PREPROCESAMIENTO PARA EL MODELO
# ============================================================

# Paso 1 — Seleccionar variables relevantes
variables_numericas = [
    'Parameters',
    'Training compute (FLOP)',
    'Training dataset size (total)',
    'Citations'
]

variables_categoricas = [
    'Domain',
    'Organization categorization',
    'Model accessibility'
]

variables_identidad = [
    'Model',
    'Organization',
    'Country (of organization)',
    'Year'
]

# Paso 2 — Crear subset con todas las variables necesarias
df_modelo = df[variables_identidad +
               variables_numericas +
               variables_categoricas].copy()

print(f"Dataset inicial: {df_modelo.shape[0]} filas")

# Paso 3 — Eliminar filas sin las variables numéricas principales
df_modelo = df_modelo.dropna(subset=['Parameters',
                                      'Training compute (FLOP)'])

print(f"Después de filtrar nulos en variables principales: {df_modelo.shape[0]} filas")

# Paso 4 — One-Hot Encoding de variables categóricas
df_encoded = pd.get_dummies(df_modelo,
                             columns=variables_categoricas,
                             drop_first=False)

print(f"Columnas después del encoding: {df_encoded.shape[1]}")
print(f"\nNuevas columnas generadas por encoding:")
nuevas = [c for c in df_encoded.columns
          if c not in variables_identidad + variables_numericas]
for col in nuevas:
    print(f"  {col}")

Dataset inicial: 3405 filas
Después de filtrar nulos en variables principales: 1212 filas
Columnas después del encoding: 40

Nuevas columnas generadas por encoding:
  Domain_3D modeling
  Domain_Audio
  Domain_Biology
  Domain_Driving
  Domain_Earth science
  Domain_Games
  Domain_Image generation
  Domain_Language
  Domain_Materials science
  Domain_Mathematics
  Domain_Medicine
  Domain_Multimodal
  Domain_Other
  Domain_Recommendation
  Domain_Robotics
  Domain_Search
  Domain_Speech
  Domain_Unknown
  Domain_Video
  Domain_Vision
  Organization categorization_Academia
  Organization categorization_Government
  Organization categorization_Industry
  Organization categorization_Research collective
  Organization categorization_Unknown
  Model accessibility_API access
  Model accessibility_Hosted access (no API)
  Model accessibility_Open weights (non-commercial)
  Model accessibility_Open weights (restricted use)
  Model accessibility_Open weights (unrestricted)
  Model accessibility

Después de filtrar nulos en variables principales trabajaremos con 1212 filas

El One-Hot Encoding generó 33 nuevas columnas binarias a partir
de las 3 variables categóricas:
- Domain generó 20 columnas — una por cada tipo de IA
- Organization categorization generó 5 columnas
- Model accessibility generó 8 columnas

En total el modelo trabajará con 37 variables:
- 4 numéricas reales
- 33 variables dummy de encoding